# Trích đặc trưng gallery ImageNet-1k train (chuẩn bị vòng 60c)

ResNet-50 (checkpoint `resnet50_imagenet1k.pth`, cùng backbone `resnet50_in1k` và autocast fp16 như bản trích tập validation trên HF).
Ảnh: cuộc thi Kaggle `imagenet-object-localization-challenge` (ILSVRC-2012). Đầu ra: các mảnh `in1k_train/feats_XXX.npy` (fp16) và `labels_XXX.npy`, kèm `manifest.json`.

**Kiểm trước khi trích:** chạy lại 512 ảnh validation đầu tiên (theo thứ tự `test_imagenet.txt` của OpenOOD đã bỏ trùng) và so với đặc trưng đã có trên HF; lệch thì dừng, ko tốn GPU vô ích.

Chạy tiếp sau khi hết phiên: gắn Output lần trước làm Input; mảnh nào đã có thì bỏ qua.

In [ ]:
print("@@CELL 0@@", flush=True)
LIMIT = None          # số ảnh train tối đa (vd 2000 để thử nhanh trên CPU); None = toàn bộ 1,28 triệu
SHARD = 64_000        # ảnh mỗi mảnh: mất phiên giữa chừng thì chỉ mất mảnh đang làm
BATCH, WORKERS = 256, 4
MIN_COS = 0.999       # ngưỡng khớp với đặc trưng cũ (cos trung bình)

In [ ]:
print("@@CELL 1@@", flush=True)
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import numpy as np

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
T0 = time.time()
def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} phút] {msg}", flush=True)

def walk(max_depth=6):
    """Duyệt /kaggle/input nhưng bỏ cây ảnh ILSVRC (~2 triệu file trên ổ mạng: rglob qua đó mất hàng giờ)."""
    for root, dirs, files in os.walk(INPUT):
        depth = len(Path(root).relative_to(INPUT).parts)
        dirs[:] = [] if depth >= max_depth else [d for d in dirs if d != "ILSVRC"]
        for f in files:
            yield Path(root) / f

FILES = sorted(walk())
def find(name):
    hits = [f for f in FILES if f.name == name]
    return hits[0] if hits else None
tick(f"duyệt input: {len(FILES)} file (ko tính ILSVRC)")

REPO = WORK / "CURE"
code_dir = next(t.parent for t in FILES if t.name == "pyproject.toml" and 'name = "cure-ood"' in t.read_text(encoding="utf-8"))
if not REPO.exists():
    shutil.copytree(code_dir, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

REF, CKPT = find("in1k_val_ref.npz"), find("resnet50_imagenet1k.pth")
SYNSET = find("LOC_synset_mapping.txt")
COMP = SYNSET.parent
TRAIN, VAL = COMP / "ILSVRC/Data/CLS-LOC/train", COMP / "ILSVRC/Data/CLS-LOC/val"
print("ref:", REF)
print("ckpt:", CKPT)
print("ảnh:", COMP)


In [ ]:
print("@@CELL 2@@", flush=True)
import torch
from torch.utils.data import DataLoader
from cure_ood.backbones import build_backbone
from cure_ood.data.extract import ListSet

dev = "cuda" if torch.cuda.is_available() else "cpu"
bb = build_backbone("resnet50_in1k", str(CKPT), device=dev)
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

@torch.inference_mode()
def run(rows, batch=BATCH, workers=WORKERS, progress=0):
    """rows: [(đường dẫn, nhãn)] → đặc trưng fp16 (n, 2048), đúng thứ tự."""
    dl = DataLoader(ListSet(rows, bb.transform), batch_size=batch, num_workers=workers, pin_memory=dev == "cuda")
    out, t = [], time.time()
    for i, (x, _) in enumerate(dl):
        f, _ = bb.encode(x.to(dev, non_blocking=True))
        out.append(f.float().cpu().numpy().astype(np.float16))
        if progress and (i + 1) % progress == 0:
            n = (i + 1) * batch
            print(f"    {n}/{len(rows)} ảnh, {n / (time.time() - t):.0f} ảnh/s", flush=True)
    return np.concatenate(out)

# Kiểm khớp với đặc trưng cũ trên 512 ảnh validation.
ref = np.load(REF)
rows = [(VAL / n, int(l)) for n, l in zip(ref["names"], ref["labels"])]
t = time.time(); new = run(rows).astype(np.float32)
a, b = new / np.linalg.norm(new, axis=1, keepdims=True), ref["feats"] / np.linalg.norm(ref["feats"], axis=1, keepdims=True)
cos = (a * b).sum(1)
check = {"cos_mean": float(cos.mean()), "cos_min": float(cos.min()), "max_abs": float(np.abs(new - ref["feats"]).max()),
         "seconds": round(time.time() - t, 1)}
print(check); tick("kiểm 512 ảnh val xong")
assert check["cos_mean"] >= MIN_COS, "đặc trưng mới ko khớp bản cũ: dừng, ko trích train"

In [ ]:
print("@@CELL 3@@", flush=True)
# Danh sách train: lớp theo thứ tự LOC_synset_mapping (= chỉ số lớp của torchvision), ảnh sắp theo tên.
wnids = [l.split()[0] for l in SYNSET.read_text().splitlines() if l.strip()]
rows = [(p, c) for c, w in enumerate(wnids) for p in sorted((TRAIN / w).iterdir())]
if LIMIT:
    rows = [rows[i] for i in np.random.default_rng(0).permutation(len(rows))[:LIMIT]]
tick(f"liệt kê {len(rows)} ảnh train, {len(wnids)} lớp")

OUT = WORK / "in1k_train"; OUT.mkdir(exist_ok=True)
prev = find("manifest.json")  # mảnh của phiên trước (Output gắn làm Input)
if prev and prev.parent != OUT:
    for f in prev.parent.glob("*.npy"):
        if not (OUT / f.name).exists():
            shutil.copy(f, OUT / f.name)
t0 = time.time()
for s in range(0, len(rows), SHARD):
    k = s // SHARD
    if (OUT / f"feats_{k:03d}.npy").exists():
        continue
    part = rows[s:s + SHARD]
    t = time.time()
    np.save(OUT / f"feats_{k:03d}.npy", run(part, progress=40))
    np.save(OUT / f"labels_{k:03d}.npy", np.array([c for _, c in part], np.int16))
    print(f"mảnh {k}: {len(part)} ảnh, {len(part) / (time.time() - t):.0f} ảnh/s, tổng {(time.time() - t0) / 60:.1f} phút", flush=True)

In [ ]:
print("@@CELL 4@@", flush=True)
manifest = {"n": len(rows), "shard": SHARD, "classes": len(wnids), "dtype": "float16", "backbone": "resnet50_in1k (amp)",
            "checkpoint": CKPT.name, "check_vs_hf_val": check, "limit": LIMIT,
            "files": [str(p.relative_to(COMP)) for p, _ in rows]}
(OUT / "manifest.json").write_text(json.dumps(manifest))
print({k: v for k, v in manifest.items() if k != "files"})
print(sorted(f.name for f in OUT.iterdir()))